<center>
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/Logos/organization_logo/organization_logo.png" width="300" alt="cognitiveclass.ai logo">
</center>


#### Import the required libraries we need for the lab.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

#### Read the dataset in the csv file from the URL


In [ ]:
boston_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv')

#### Add your code below following the instructions given in the course to complete the peer graded assignment

---

## Before we dive into numbers — what are we actually looking at?

This dataset is really a story about **506 Boston neighborhoods** in the late 1970s, and what made a family's home worth more or less money. Behind every row is a real town — its schools, its factories, its distance from the city, whether it sat near the Charles River. Our job as data scientists is to translate that story into evidence: what actually moved home prices, and what didn't?

We'll walk through this the way a curious analyst would — asking a question, looking at the picture, checking it with statistics, and then explaining what it *means* for the people living there.


### 1. What does a "typical" home look like? — Boxplot for MEDV

**The human question:** If you had to describe home values in these towns to a friend in one picture, what would you show them? A boxplot tells us the typical range, and — just as importantly — which towns were outliers, either surprisingly cheap or surprisingly expensive.


In [ ]:
pyplot.figure(figsize=(8, 5))
sns.boxplot(y=boston_df['MEDV'])
pyplot.title('Median Value of Owner-Occupied Homes (MEDV)')
pyplot.ylabel('Median value ($1000s)')
pyplot.show()

**What this tells us:** Most homes cluster between roughly \$17,000 and \$25,000 (in 1970s dollars), with a median around \$21,200. But notice the cluster of dots at the very top of the chart — a group of towns capped at exactly \$50,000. That's not a coincidence; the original survey top-coded high-value homes at that number, so we're likely underestimating how valuable the priciest neighborhoods really were. A good analyst always asks *why* the data looks the way it does, not just *what* it looks like.


### 2. How many towns actually border the Charles River? — Bar plot for CHAS

**The human question:** Riverside living is often considered a premium — but how rare is it in this dataset? If almost no towns touch the river, we should be cautious about generalizing from them.


In [ ]:
chas_counts = boston_df['CHAS'].value_counts()

pyplot.figure(figsize=(6, 5))
sns.barplot(x=chas_counts.index.astype(str), y=chas_counts.values)
pyplot.title('Number of Towns Bounded by the Charles River')
pyplot.xlabel('Bounds Charles River (0 = No, 1 = Yes)')
pyplot.ylabel('Number of towns')
pyplot.show()

**What this tells us:** Out of 506 towns, only **35 (about 7%)** actually border the Charles River, while **471 (about 93%)** don't. Riverside living really is the exception, not the rule — which is exactly why, later, we'll want a formal test rather than just eyeballing averages, since we're comparing a small group against a much larger one.


### 3. Does the age of a town's housing stock relate to its price? — Boxplot for AGE (grouped)

**The human question:** Older neighborhoods can go two ways — they can feel historic and charming, or run-down and dated. Let's group towns by how old their homes typically are, and see if that shows up in price.

We'll split `AGE` (percentage of owner-occupied units built before 1940) into three human-readable buckets:
- **35 years and younger**
- **Between 35 and 70 years**
- **70 years and older**


In [ ]:
age_labels = ['35 years and younger', 'between 35 and 70 years', '70 years and older']
age_bins = [0, 35, 70, 100]
boston_df['age_group'] = pd.cut(boston_df['AGE'], bins=age_bins, labels=age_labels, include_lowest=True)

pyplot.figure(figsize=(9, 5))
sns.boxplot(x='age_group', y='MEDV', data=boston_df)
pyplot.title('Median Home Value by Housing Age Group')
pyplot.xlabel('Age of housing stock')
pyplot.ylabel('Median value ($1000s)')
pyplot.show()

**What this tells us:** Towns with newer housing (35 years and younger) tend to have noticeably higher and more tightly-clustered home values, while towns dominated by older housing (70+ years) show lower medians and a wider spread — including some real low outliers. In plain terms: **older housing stock, on average, tends to sit in lower-value neighborhoods** in this dataset, though it's not a hard rule — some older towns still hold their value well.


### 4. Do industrial towns have dirtier air? — Scatter plot for NOX vs INDUS

**The human question:** It seems intuitive that more industry means more pollution — but does the data actually back that up, or is it just a stereotype?


In [ ]:
pyplot.figure(figsize=(8, 5))
sns.regplot(x='INDUS', y='NOX', data=boston_df, scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
pyplot.title('Nitric Oxide Concentration vs. Proportion of Industrial Land')
pyplot.xlabel('INDUS - proportion of non-retail business acres per town')
pyplot.ylabel('NOX - nitric oxide concentration (parts per 10 million)')
pyplot.show()

**What this tells us:** Yes — the stereotype holds up here. As the proportion of industrial/non-retail business land in a town rises, nitric oxide levels climb right along with it. It's a real, visible trend, and we'll confirm exactly how strong it is with Pearson's correlation below.


### 5. How crowded are the classrooms? — Histogram for PTRATIO

**The human question:** Pupil-teacher ratio is a proxy for school quality and resources. What does the "typical" Boston-area classroom look like in this data, and are most towns similar, or is there a lot of variation?


In [ ]:
pyplot.figure(figsize=(8, 5))
pyplot.hist(boston_df['PTRATIO'], bins=15, edgecolor='black')
pyplot.title('Distribution of Pupil-Teacher Ratio')
pyplot.xlabel('Pupil-Teacher Ratio')
pyplot.ylabel('Number of towns')
pyplot.show()

**What this tells us:** The distribution is **left-skewed**, with a heavy concentration of towns sitting around a ratio of roughly 20 students per teacher — and a noticeably tall spike right around there. There's a smaller group of towns with much lower ratios (better-resourced schools, likely fewer students per teacher). So most families in this dataset experienced fairly similar, moderately crowded classrooms, with a minority enjoying a real advantage.


### 6. Is living near the Charles River actually worth more? — T-test

**The human question:** "Riverside" sounds premium, but is that reflected in real home prices, or is it just marketing?

**Hypotheses:**
- H₀: There is no significant difference in median home values (MEDV) between towns that border the river and those that don't.
- H₁: There **is** a significant difference in median home values between the two groups.


In [ ]:
river = boston_df[boston_df['CHAS'] == 1]['MEDV']
no_river = boston_df[boston_df['CHAS'] == 0]['MEDV']

t_stat, p_value = scipy.stats.ttest_ind(no_river, river, equal_var=False)
print("T-statistic:", t_stat)
print("P-value:", p_value)

**What this tells us:** With a p-value of about **0.0036** (well below 0.05), we **reject the null hypothesis**. There genuinely is a statistically significant difference — towns bordering the Charles River tend to have higher median home values. So the "riverside premium" isn't just a nice story; it's backed by the numbers. Just remember: only 35 of 506 towns are riverside, so this is a real but relatively rare advantage.


### 7. Does housing age really affect price across the board? — ANOVA

**The human question:** The boxplot in Question 3 hinted at a pattern — now let's formally check whether the differences we saw between the three age groups are real, or could have happened by chance.

**Hypotheses:**
- H₀: There is no significant difference in MEDV across the three age groups.
- H₁: At least one age group has a significantly different median home value.


In [ ]:
lm = ols('MEDV ~ C(age_group)', data=boston_df).fit()
anova_table = sm.stats.anova_lm(lm, typ=2)
print(anova_table)

**What this tells us:** The p-value here is astronomically small (well under 0.001), so we **reject the null hypothesis**. The age of a town's housing stock genuinely relates to how much homes are worth — this isn't a coincidence in the data. Combined with what we saw in the boxplot, the story is consistent: **newer housing stock tends to command a real price premium.**


### 8. How strong is the industry–pollution link, exactly? — Pearson Correlation (NOX vs INDUS)

**The human question:** We saw the trend visually in Question 4 — now let's put a precise number on how strong that relationship really is.


In [ ]:
corr, p_value = scipy.stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])
print("Pearson correlation coefficient (r):", corr)
print("P-value:", p_value)

**What this tells us:** r ≈ **0.76**, with a p-value essentially at zero. That's a **strong, statistically significant positive correlation** — about 58% of the variation in nitric oxide levels can be explained just by how industrial a town is. In human terms: if you wanted to predict which Boston-area towns had the worst air quality in this era, "how much industry is here?" would be one of your best single clues.


### 9. Does living farther from job centers hurt home values? — Regression Analysis (DIS vs MEDV)

**The human question:** Commuting distance matters to real families — does being far from Boston's employment centers actually show up as lower (or higher) home prices, and how much of the story does distance alone explain?

**Hypotheses:**
- H₀: DIS has no significant effect on MEDV (slope = 0).
- H₁: DIS has a significant effect on MEDV.


In [ ]:
X = sm.add_constant(boston_df['DIS'])
model = sm.OLS(boston_df['MEDV'], X).fit()
print(model.summary())

**What this tells us:** The relationship is statistically significant (p < 0.001) — and, perhaps surprisingly, the coefficient on DIS is **positive**: each extra unit of weighted distance from Boston's employment centers is associated with roughly a **\$1,090 increase** in median home value. This runs counter to a simple "closer to jobs = more valuable" assumption; in this dataset, towns farther from the employment centers include some of the more spacious, higher-value suburbs. That said, the R² here is small (about 6%), meaning distance alone explains only a modest slice of the full pricing picture — plenty of other factors (like industry, schools, and river access, which we already explored) matter more.


## Bringing it all together

If you zoom out from all nine questions, a coherent human story emerges about Boston-area housing in the late 1970s:

- **Where you lived mattered a lot.** Riverside towns commanded a real premium, and newer housing stock was worth meaningfully more than older housing stock.
- **Industry had a visible cost.** Towns with more industrial land had noticeably worse air quality — a trade-off that likely showed up in who chose to live there and at what price.
- **School resources varied, but not wildly.** Most towns had broadly similar pupil-teacher ratios, with a smaller group enjoying a real advantage.
- **Distance from jobs wasn't the dominant factor** it might seem — it explained only a small part of price variation once you consider everything else going on in a neighborhood.

Every one of these findings started as a simple visual and ended as a tested, defensible claim — which is really the whole craft of data science: turning "it looks like..." into "the evidence shows...".
